In [2]:
import numpy as np
from pathlib import Path

In [4]:
# Processed data path

data_dir = Path("processed_data")

print("Processed data directory:", data_dir.resolve())
print("Directory exists:", data_dir.exists())

Processed data directory: E:\data of brain projec\data of brain project\EEG_project\processed_data
Directory exists: True


In [5]:
alpha_asymmetry = np.load(data_dir / "alpha_asymmetry.npy")
alpha_power = np.load(data_dir / "alpha_power.npy")

beta_asymmetry = np.load(data_dir / "beta_asymmetry.npy")
beta_power = np.load(data_dir / "beta_power.npy")

correlation = np.load(data_dir / "correlation.npy")
mi = np.load(data_dir / "mi.npy")

print("Alpha Asymmetry:", alpha_asymmetry.shape)
print("Alpha Power:", alpha_power.shape)
print("Beta Asymmetry:", beta_asymmetry.shape)
print("Beta Power:", beta_power.shape)
print("Correlation:", correlation.shape)
print("MI:", mi.shape)

Alpha Asymmetry: (85330, 7)
Alpha Power: (85330, 14)
Beta Asymmetry: (85330, 7)
Beta Power: (85330, 14)
Correlation: (85330, 14, 14)
MI: (85330, 14, 14)


In [ ]:
# Check NaN and Inf values
features = {
    "Alpha Asymmetry": alpha_asymmetry,
    "Alpha Power": alpha_power,
    "Beta Asymmetry": beta_asymmetry,
    "Beta Power": beta_power,
    "Correlation": correlation,
    "MI": mi
}

for name, feature in features.items():
    print(
        f"{name}: "
        f"NaN = {np.isnan(feature).sum()}, "
        f"Inf = {np.isinf(feature).sum()}"
    )

Alpha Asymmetry: NaN = 0, Inf = 0
Alpha Power: NaN = 0, Inf = 0
Beta Asymmetry: NaN = 0, Inf = 0
Beta Power: NaN = 0, Inf = 0
Correlation: NaN = 0, Inf = 0
MI: NaN = 0, Inf = 0


In [ ]:
# Check feature range
for name, feature in features.items():
    print(f"\n{name}")
    print("  Min :", np.min(feature))
    print("  Max :", np.max(feature))
    print("  Mean:", np.mean(feature))
    print("  Std :", np.std(feature))


Alpha Asymmetry
  Min : -24.94967
  Max : 24.894274
  Mean: -1.3043098
  Std : 1.674336

Alpha Power
  Min : 4.5528534e-26
  Max : 662052.75
  Mean: 139.82608
  Std : 3339.3447

Beta Asymmetry
  Min : -25.397486
  Max : 26.211737
  Mean: -1.0205685
  Std : 1.5198295

Beta Power
  Min : 7.737298e-29
  Max : 663713.4
  Mean: 5370.5796
  Std : 32078.121

Correlation
  Min : -0.9934373
  Max : 1.0
  Mean: 0.44027072
  Std : 0.3680898

MI
  Min : 0.0
  Max : 3.9112108
  Mean: 0.29891795
  Std : 0.40816706


In [8]:
# ============================================================
# Flatten matrix-based features
# ============================================================

correlation_flat = correlation.reshape(
    correlation.shape[0], -1
)

mi_flat = mi.reshape(
    mi.shape[0], -1
)

print("Original Correlation shape:", correlation.shape)
print("Flattened Correlation shape:", correlation_flat.shape)

print("\nOriginal MI shape:", mi.shape)
print("Flattened MI shape:", mi_flat.shape)

Original Correlation shape: (85330, 14, 14)
Flattened Correlation shape: (85330, 196)

Original MI shape: (85330, 14, 14)
Flattened MI shape: (85330, 196)


In [ ]:
# Extract unique connections from symmetric matrices

n_channels = correlation.shape[1]

upper_triangle = np.triu_indices(
    n_channels,
    k=1
)

correlation_features = correlation[
    :, upper_triangle[0], upper_triangle[1]
]

mi_features = mi[
    :, upper_triangle[0], upper_triangle[1]
]

print("Correlation features shape:", correlation_features.shape)
print("MI features shape:", mi_features.shape)

Correlation features shape: (85330, 91)
MI features shape: (85330, 91)


In [ ]:
# Check prepared feature statistics

prepared_features = {
    "Alpha Power": alpha_power,
    "Beta Power": beta_power,
    "Alpha Asymmetry": alpha_asymmetry,
    "Beta Asymmetry": beta_asymmetry,
    "Correlation": correlation_features,
    "MI": mi_features
}

for name, feature in prepared_features.items():
    print(
        f"{name}: "
        f"mean={np.mean(feature):.6f}, "
        f"std={np.std(feature):.6f}, "
        f"min={np.min(feature):.6f}, "
        f"max={np.max(feature):.6f}"
    )

Alpha Power: mean=139.826080, std=3339.344727, min=0.000000, max=662052.750000
Beta Power: mean=5370.579590, std=32078.121094, min=0.000000, max=663713.375000
Alpha Asymmetry: mean=-1.304310, std=1.674336, min=-24.949671, max=24.894274
Beta Asymmetry: mean=-1.020568, std=1.519830, min=-25.397486, max=26.211737
Correlation: mean=0.397215, std=0.346351, min=-0.993437, max=0.999986
MI: mean=0.321912, std=0.414745, min=0.000000, max=3.911211


In [11]:
# ============================================================
# Check Power distribution
# ============================================================

print("Alpha Power percentiles:")
print(np.percentile(
    alpha_power,
    [0, 25, 50, 75, 90, 95, 99, 99.9, 100]
))

print("\nBeta Power percentiles:")
print(np.percentile(
    beta_power,
    [0, 25, 50, 75, 90, 95, 99, 99.9, 100]
))

Alpha Power percentiles:
[4.55285340e-26 4.55518329e+00 1.13160276e+01 2.52451339e+01
 5.80737625e+01 1.19456822e+02 2.15140893e+03 1.61931090e+04
 6.62052750e+05]

Beta Power percentiles:
[7.73729776e-29 7.96064579e+00 1.62444830e+01 3.80014620e+01
 3.04047464e+02 1.58970783e+04 2.37914950e+05 3.06345897e+05
 6.63713375e+05]


In [12]:
# ============================================================
# Log transform Power features
# ============================================================

alpha_power_log = np.log1p(alpha_power)
beta_power_log = np.log1p(beta_power)

print("Alpha Power - before:")
print(
    "Mean:", np.mean(alpha_power),
    "Std:", np.std(alpha_power),
    "Max:", np.max(alpha_power)
)

print("\nAlpha Power - after log1p:")
print(
    "Mean:", np.mean(alpha_power_log),
    "Std:", np.std(alpha_power_log),
    "Max:", np.max(alpha_power_log)
)

print("\nBeta Power - before:")
print(
    "Mean:", np.mean(beta_power),
    "Std:", np.std(beta_power),
    "Max:", np.max(beta_power)
)

print("\nBeta Power - after log1p:")
print(
    "Mean:", np.mean(beta_power_log),
    "Std:", np.std(beta_power_log),
    "Max:", np.max(beta_power_log)
)

Alpha Power - before:
Mean: 139.82608 Std: 3339.3447 Max: 662052.75

Alpha Power - after log1p:
Mean: 2.5393171 Std: 1.4529135 Max: 13.403102

Beta Power - before:
Mean: 5370.5796 Std: 32078.121 Max: 663713.4

Beta Power - after log1p:
Mean: 3.343439 Std: 2.309033 Max: 13.405607


In [ ]:
# Standardize prepared features

from sklearn.preprocessing import StandardScaler

scalers = {}

scaled_features = {}

for name, feature in {
    "Alpha Power": alpha_power_log,
    "Beta Power": beta_power_log,
    "Alpha Asymmetry": alpha_asymmetry,
    "Beta Asymmetry": beta_asymmetry,
    "Correlation": correlation_features,
    "MI": mi_features
}.items():

    scaler = StandardScaler()

    scaled = scaler.fit_transform(feature)

    scalers[name] = scaler
    scaled_features[name] = scaled.astype(np.float32)

    print(
        f"{name}: "
        f"shape={scaled_features[name].shape}, "
        f"mean={np.mean(scaled_features[name]):.6f}, "
        f"std={np.std(scaled_features[name]):.6f}"
    )

Alpha Power: shape=(85330, 14), mean=-0.000000, std=1.000000
Beta Power: shape=(85330, 14), mean=-0.000000, std=1.000000
Alpha Asymmetry: shape=(85330, 7), mean=0.000000, std=1.000000
Beta Asymmetry: shape=(85330, 7), mean=0.000000, std=1.000000
Correlation: shape=(85330, 91), mean=-0.000000, std=1.000000
MI: shape=(85330, 91), mean=-0.000000, std=1.000000


In [15]:
# Concatenate all standardized features

fused_features = np.concatenate(
    [
        scaled_features["Alpha Power"],
        scaled_features["Beta Power"],
        scaled_features["Alpha Asymmetry"],
        scaled_features["Beta Asymmetry"],
        scaled_features["Correlation"],
        scaled_features["MI"]
    ],
    axis=1
)

print("Fused features shape:", fused_features.shape)

Fused features shape: (85330, 224)


NaN: 0
Inf: 0
Mean: 3.2762546e-08
Std: 0.9999998


In [17]:
# Feature column mapping

feature_names = []

feature_names += [f"alpha_power_{i}" for i in range(14)]
feature_names += [f"beta_power_{i}" for i in range(14)]

feature_names += [
    f"alpha_asymmetry_{i}" for i in range(7)
]

feature_names += [
    f"beta_asymmetry_{i}" for i in range(7)
]

feature_names += [
    f"correlation_{i}" for i in range(91)
]

feature_names += [
    f"mi_{i}" for i in range(91)
]

print("Number of feature names:", len(feature_names))
print("First 10:", feature_names[:10])
print("Last 10:", feature_names[-10:])

Number of feature names: 224
First 10: ['alpha_power_0', 'alpha_power_1', 'alpha_power_2', 'alpha_power_3', 'alpha_power_4', 'alpha_power_5', 'alpha_power_6', 'alpha_power_7', 'alpha_power_8', 'alpha_power_9']
Last 10: ['mi_81', 'mi_82', 'mi_83', 'mi_84', 'mi_85', 'mi_86', 'mi_87', 'mi_88', 'mi_89', 'mi_90']


In [20]:
# Load labels and metadata

valence = np.load(data_dir / "valence.npy")
arousal = np.load(data_dir / "arousal.npy")
dominance = np.load(data_dir / "dominance.npy")

subjects = np.load(data_dir / "subjects.npy")
trials = np.load(data_dir / "trials.npy")

print("Valence:", valence.shape)
print("Arousal:", arousal.shape)
print("Dominance:", dominance.shape)
print("Subjects:", subjects.shape)
print("Trials:", trials.shape)

Valence: (85330,)
Arousal: (85330,)
Dominance: (85330,)
Subjects: (85330,)
Trials: (85330,)


In [21]:
# Check dataset metadata

print("Number of unique subjects:", len(np.unique(subjects)))
print("Number of unique trials:", len(np.unique(trials)))

print("Subjects:", np.unique(subjects))
print("Trials:", np.unique(trials))

print("\nFirst 10 samples:")
for i in range(10):
    print(
        i,
        "Subject:", subjects[i],
        "Trial:", trials[i],
        "Valence:", valence[i],
        "Arousal:", arousal[i],
        "Dominance:", dominance[i]
    )

Number of unique subjects: 23
Number of unique trials: 18
Subjects: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]
Trials: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]

First 10 samples:
0 Subject: 1 Trial: 1 Valence: 4.0 Arousal: 3.0 Dominance: 2.0
1 Subject: 1 Trial: 1 Valence: 4.0 Arousal: 3.0 Dominance: 2.0
2 Subject: 1 Trial: 1 Valence: 4.0 Arousal: 3.0 Dominance: 2.0
3 Subject: 1 Trial: 1 Valence: 4.0 Arousal: 3.0 Dominance: 2.0
4 Subject: 1 Trial: 1 Valence: 4.0 Arousal: 3.0 Dominance: 2.0
5 Subject: 1 Trial: 1 Valence: 4.0 Arousal: 3.0 Dominance: 2.0
6 Subject: 1 Trial: 1 Valence: 4.0 Arousal: 3.0 Dominance: 2.0
7 Subject: 1 Trial: 1 Valence: 4.0 Arousal: 3.0 Dominance: 2.0
8 Subject: 1 Trial: 1 Valence: 4.0 Arousal: 3.0 Dominance: 2.0
9 Subject: 1 Trial: 1 Valence: 4.0 Arousal: 3.0 Dominance: 2.0


In [22]:
# Count windows per subject and trial

unique_pairs, counts = np.unique(
    np.column_stack((subjects, trials)),
    axis=0,
    return_counts=True
)

print("Number of subject-trial combinations:", len(unique_pairs))

print("\nFirst 10 combinations:")
for pair, count in zip(unique_pairs[:10], counts[:10]):
    print(
        f"Subject {pair[0]}, Trial {pair[1]}: "
        f"{count} windows"
    )

print("\nWindow count statistics:")
print("Min:", np.min(counts))
print("Max:", np.max(counts))
print("Mean:", np.mean(counts))
print("Unique window counts:", np.unique(counts))

Number of subject-trial combinations: 414

First 10 combinations:
Subject 1, Trial 1: 198 windows
Subject 1, Trial 2: 130 windows
Subject 1, Trial 3: 347 windows
Subject 1, Trial 4: 165 windows
Subject 1, Trial 5: 135 windows
Subject 1, Trial 6: 189 windows
Subject 1, Trial 7: 191 windows
Subject 1, Trial 8: 393 windows
Subject 1, Trial 9: 144 windows
Subject 1, Trial 10: 66 windows

Window count statistics:
Min: 66
Max: 393
Mean: 206.11111111111111
Unique window counts: [ 66  95 130 135 144 165 169 180 185 189 191 194 198 255 307 347 367 393]


In [23]:
# Check label consistency within each trial

labels = np.column_stack([
    valence,
    arousal,
    dominance
])

groups = np.column_stack([
    subjects,
    trials
])

unique_groups = np.unique(groups, axis=0)

inconsistent_groups = []

for subject_id, trial_id in unique_groups:

    mask = (
        (subjects == subject_id) &
        (trials == trial_id)
    )

    unique_labels = np.unique(
        labels[mask],
        axis=0
    )

    if len(unique_labels) != 1:
        inconsistent_groups.append(
            (subject_id, trial_id, unique_labels)
        )

print("Total subject-trial groups:", len(unique_groups))
print("Inconsistent groups:", len(inconsistent_groups))

if inconsistent_groups:
    print("\nFirst inconsistent groups:")
    for group in inconsistent_groups[:5]:
        print(group)

Total subject-trial groups: 414
Inconsistent groups: 0


In [ ]:
# in this case we have load data => Nan/inf check => correlation and MI → 91 unique connections
# then log1p of power => standardScaler => concatenate => label alignment and label consistency

In [ ]:
#split dataset to train , test and validation

from sklearn.model_selection import train_test_split

# Split subjects into train and temporary sets

unique_subjects = np.unique(subjects)

train_subjects, temp_subjects = train_test_split(
    unique_subjects,
    test_size=7,
    random_state=42,
    shuffle=True
)

# Split temporary subjects into validation and test

val_subjects, test_subjects = train_test_split(
    temp_subjects,
    test_size=4,
    random_state=42,
    shuffle=True
)

print("Train subjects:", np.sort(train_subjects))
print("Validation subjects:", np.sort(val_subjects))
print("Test subjects:", np.sort(test_subjects))

print("\nNumber of subjects:")
print("Train:", len(train_subjects))
print("Validation:", len(val_subjects))
print("Test:", len(test_subjects))

Train subjects: [ 3  4  5  6  7  8 11 12 14 15 17 19 20 21 22 23]
Validation subjects: [ 2  9 18]
Test subjects: [ 1 10 13 16]

Number of subjects:
Train: 16
Validation: 3
Test: 4


In [ ]:
# Create indices for each split and  check out 

train_mask = np.isin(subjects, train_subjects)
val_mask = np.isin(subjects, val_subjects)
test_mask = np.isin(subjects, test_subjects)

train_indices = np.where(train_mask)[0]
val_indices = np.where(val_mask)[0]
test_indices = np.where(test_mask)[0]

print("Train windows:", len(train_indices))
print("Validation windows:", len(val_indices))
print("Test windows:", len(test_indices))

print("\nTotal windows:", len(train_indices) + len(val_indices) + len(test_indices))

Train windows: 59360
Validation windows: 11130
Test windows: 14840

Total windows: 85330


In [26]:
# Create train, validation, and test datasets

X_train = fused_features[train_indices]
X_val = fused_features[val_indices]
X_test = fused_features[test_indices]

y_valence_train = valence[train_indices]
y_valence_val = valence[val_indices]
y_valence_test = valence[test_indices]

y_arousal_train = arousal[train_indices]
y_arousal_val = arousal[val_indices]
y_arousal_test = arousal[test_indices]

y_dominance_train = dominance[train_indices]
y_dominance_val = dominance[val_indices]
y_dominance_test = dominance[test_indices]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nValence:")
print(y_valence_train.shape, y_valence_val.shape, y_valence_test.shape)

print("\nArousal:")
print(y_arousal_train.shape, y_arousal_val.shape, y_arousal_test.shape)

print("\nDominance:")
print(y_dominance_train.shape, y_dominance_val.shape, y_dominance_test.shape)

X_train: (59360, 224)
X_val: (11130, 224)
X_test: (14840, 224)

Valence:
(59360,) (11130,) (14840,)

Arousal:
(59360,) (11130,) (14840,)

Dominance:
(59360,) (11130,) (14840,)


In [ ]:
# Check label distribution

for name, train_y, val_y, test_y in [
    ("Valence", y_valence_train, y_valence_val, y_valence_test),
    ("Arousal", y_arousal_train, y_arousal_val, y_arousal_test),
    ("Dominance", y_dominance_train, y_dominance_val, y_dominance_test)
]:
    print(f"\n{name}")

    print("Train:")
    print(np.unique(train_y, return_counts=True))

    print("Validation:")
    print(np.unique(val_y, return_counts=True))

    print("Test:")
    print(np.unique(test_y, return_counts=True))


# Note:
# Arousal and Dominance have no class 1 samples in the validation set.
# This is a result of the subject-level split used to prevent data leakage.
# We keep this split because avoiding subject overlap between train and
# validation/test sets is more important than forcing identical class
# distributions across the splits.


Valence
Train:
(array([1., 2., 3., 4., 5.], dtype=float32), array([11848, 11088, 13068, 13905,  9451]))
Validation:
(array([1., 2., 3., 4., 5.], dtype=float32), array([1939, 3035, 1963, 2574, 1619]))
Test:
(array([1., 2., 3., 4., 5.], dtype=float32), array([3332, 2131, 1855, 4531, 2991]))

Arousal
Train:
(array([1., 2., 3., 4., 5.], dtype=float32), array([ 2723, 11482, 17648, 21045,  6462]))
Validation:
(array([2., 3., 4., 5.], dtype=float32), array([1176, 3022, 4483, 2449]))
Test:
(array([1., 2., 3., 4., 5.], dtype=float32), array([ 761, 3680, 3930, 4366, 2103]))

Dominance
Train:
(array([1., 2., 3., 4., 5.], dtype=float32), array([ 1741, 10406, 17201, 20293,  9719]))
Validation:
(array([2., 3., 4., 5.], dtype=float32), array([1290, 2739, 5265, 1836]))
Test:
(array([1., 2., 3., 4., 5.], dtype=float32), array([ 524, 3171, 2410, 5124, 3611]))


In [31]:
# Save final split datasets

np.save(split_dir / "X_train.npy", X_train)
np.save(split_dir / "X_val.npy", X_val)
np.save(split_dir / "X_test.npy", X_test)

np.save(split_dir / "y_valence_train.npy", y_valence_train)
np.save(split_dir / "y_valence_val.npy", y_valence_val)
np.save(split_dir / "y_valence_test.npy", y_valence_test)

np.save(split_dir / "y_arousal_train.npy", y_arousal_train)
np.save(split_dir / "y_arousal_val.npy", y_arousal_val)
np.save(split_dir / "y_arousal_test.npy", y_arousal_test)

np.save(split_dir / "y_dominance_train.npy", y_dominance_train)
np.save(split_dir / "y_dominance_val.npy", y_dominance_val)
np.save(split_dir / "y_dominance_test.npy", y_dominance_test)

print("Final split datasets saved.")

Final split datasets saved.
